In [3]:
import pandas as pd
import numpy as np
import os

def load_wesad_data(directory):
    data = {}
    for participant_id in os.listdir(directory):
        participant_path = os.path.join(directory, participant_id)
        if os.path.isdir(participant_path):
            participant_data = {}
            for file_name in os.listdir(participant_path):
                file_path = os.path.join(participant_path, file_name)
                if file_name.endswith('.pkl'):
                    participant_data[file_name.split('.')[0]] = pd.read_pickle(file_path)
            data[participant_id] = participant_data
    return data

wesad_data = load_wesad_data('C:/Users/rusha/Desktop/Uni_Freiburg_Notes/MDD/WESAD')

In [4]:
def preprocess_data(wesad_data):
    processed_data = {}
    for participant_id, participant_data in wesad_data.items():
        processed_participant_data = {}
        for sensor, data in participant_data.items():
            if isinstance(data, pd.DataFrame):
                data = data.fillna(data.mean())
                data = (data - data.min()) / (data.max() - data.min())
                processed_participant_data[sensor] = data
        processed_data[participant_id] = processed_participant_data
    return processed_data

preprocessed_wesad_data = preprocess_data(wesad_data)

In [7]:
'''IGNORE FOR NOW'''


from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.2, random_state=42)

# Train the Random Forest classifier
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Predict on the test set
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy}')
print(classification_report(y_test, y_pred))

ValueError: With n_samples=0, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

In [22]:
# Access the data for subject 'S10'
subject_s10_data = wesad_data['S10']

# Print out the structure of subject_s10_data to understand its content
print("Structure of data for subject S10:")
print(subject_s10_data)


Structure of data for subject S10:
{'S10': {'signal': {'chest': {'ACC': array([[ 1.12779999,  0.15199995,  0.34159994],
       [ 1.09319997,  0.18879998,  0.29219997],
       [ 1.03539991,  0.20940006,  0.18579996],
       ...,
       [ 0.89419997,  0.03380001, -0.21460003],
       [ 0.89499998,  0.03419995, -0.21820003],
       [ 0.89639997,  0.03260005, -0.22140002]]), 'ECG': array([[-1.33369446],
       [-1.32774353],
       [-1.32206726],
       ...,
       [ 0.53050232],
       [ 0.53375244],
       [ 0.54057312]]), 'EMG': array([[-0.01368713],
       [-0.02192688],
       [-0.00901794],
       ...,
       [ 0.00654602],
       [-0.00141907],
       [-0.00814819]]), 'EDA': array([[0.71601868],
       [0.7144928 ],
       [0.71563721],
       ...,
       [1.70440674],
       [1.74827576],
       [1.72462463]]), 'Temp': array([[33.69586 ],
       [33.741333],
       [33.71707 ],
       ...,
       [35.020447],
       [34.932495],
       [34.944824]], dtype=float32), 'Resp': array([[

In [31]:
# Access the data for subject 'S10'
subject_s10_data = wesad_data['S10']
s10_data_details = subject_s10_data['S10']
labels_s10 = s10_data_details['label']
print("labels for subject S10:")
print(labels_s10)

Labels for subject S10:
[0 0 0 ... 0 0 0]


In [53]:
len(s10_data_details["signal"]["wrist"]["ACC"])

175872

In [36]:
features_s10 = s10_data_details['signal']['wrist']['ACC']
labels_s10 = s10_data_details['label']

In [37]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
features_s10_normalized = scaler.fit_transform(features_s10)

In [38]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(features_s10_normalized, labels_s10, test_size=0.2, random_state=42)

ValueError: Found input variables with inconsistent numbers of samples: [175872, 3847200]

In [39]:
print("Shape of features_s10:", features_s10.shape)
print("Shape of labels_s10:", labels_s10.shape)

Shape of features_s10: (175872, 3)
Shape of labels_s10: (3847200,)


In [40]:
# ensure number of rows in features_s10 matches no. of labels
n_samples = min(features_s10.shape[0], len(labels_s10))

# trim features and labels to have same no. of samples
features_s10_trimmed = features_s10[:n_samples]
labels_s10_trimmed = labels_s10[:n_samples]

X_train, X_test, y_train, y_test = train_test_split(features_s10_trimmed, labels_s10_trimmed, test_size=0.2, random_state=42)

In [44]:
X_train

array([[ 38.,   1.,  49.],
       [ 38.,   3.,  50.],
       [ 57., -10., -24.],
       ...,
       [ 16.,  13.,  59.],
       [ 22., -21.,  55.],
       [ 53., -24., -24.]])

In [54]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"accuracy: {accuracy:.2f}")
print("classification report:")
print(report)

accuracy: 0.94
classification report:
              precision    recall  f1-score   support

           0       0.92      0.92      0.92     12910
           1       0.95      0.95      0.95     22265

    accuracy                           0.94     35175
   macro avg       0.94      0.94      0.94     35175
weighted avg       0.94      0.94      0.94     35175

